    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.


    – Task 2b: Implement a program which, given (a) a query imageID or image file and 
    (b) positive integer k, identifies and lists k most likely matching labels, along 
    with their scores, under the RESNET50 neural network model.

In [1]:
# Import Libraries 
import pandas as pd
import numpy as np
import torch
import torchvision
import PIL
from pathlib import Path
from tqdm import tqdm
import scipy
import sklearn.cluster
import sklearn.metrics
import sys
import math
from time import sleep
import gc

# Custom Functions
from utils.query_input_processor import get_query_image
from utils.image_utils import convert_to_rgb

c:\Users\Panik\miniconda3\envs\cs515\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\Panik\miniconda3\envs\cs515\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
c:\Users\Panik\miniconda3\envs\cs515\lib\site-packages\numpy\.libs\libopenblas64__v0.3.23-246-g3d31191b-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:
# Research References:
# https://saturncloud.io/blog/how-to-check-if-pytorch-is-using-the-gpu/
# https://saturncloud.io/blog/how-to-train-a-pytorch-model-on-gpu/
# https://docs.scipy.org/doc/scipy/reference/spatial.distance.html
# https://learnopencv.com/image-classification-using-transfer-learning-in-pytorch/
# https://pytorch.org/docs/master/generated/torch.topk.html
# https://pytorch.org/vision/stable/models.html

In [3]:
# These utility functions are unique to task 2b
# Using convert_to_rgb, get_query_image from utility folder

# These function gets the device to use as the goal of 
# the code is to use GPU if possible
def getDevice():
    torchDevice = "cpu"
    if torch.cuda.is_available():
        torchDevice = "cuda"

    print("Using Device: ", torchDevice)
    return torchDevice


# This function converts the image to 224x224 as required by
# the ResNet model, then transforms it with the ResNet weight,
# and then gets the ResNet Input
def resNetImageTransformer(image, resNetModel, resNetWeight, torchDevice):
    # Convert Non-RGB to RGB
    if image.mode != "RGB":
        image = convert_to_rgb(image)
        
    # Resize Image
    imageCopy = image.resize(size=(224, 224))
    
    # Put image into model and get results
    imageTransform = resNetWeight.transforms()(imageCopy).unsqueeze(0).to(torchDevice)
    result = resNetModel(imageTransform).squeeze(0).softmax(0).to(torchDevice)

    return result.detach().cpu().numpy()


# This function gets the name of the label id given in the database
def getCalTechCategory(caltechDB, label_id):
    return caltechDB.annotation_categories[label_id]

# This function setsup the ResNet Model
def getResNetModel(torchDevice):
    resNetWeight = torchvision.models.ResNet50_Weights.DEFAULT
    resNetModel = torchvision.models.resnet50(progress=True, weights=resNetWeight).to(torchDevice)
    for parameter in resNetModel.parameters():
        parameter.requires_grad = False
    resNetModel.zero_grad(set_to_none=True)
    resNetModel.eval()

    return resNetWeight, resNetModel


In [23]:
# Function takes an image and finds the k similar labels using kmeans

# Getting optimal cluster algorithm is derived from these resources:
# https://medium.com/analytics-vidhya/how-to-determine-the-optimal-k-for-k-means-708505d204eb
# https://www.geeksforgeeks.org/ml-determine-the-optimal-value-of-k-in-k-means-clustering/

# Clustering and Parameters used from with modifications to remove randomization and use auto optimizers: 
# https://www.analyticsvidhya.com/blog/2021/01/a-simple-guide-to-centroid-based-clustering-with-python-code/

# Caleb - Explaination of Approach as many itterations and attempts are not show otherwise notebook will become messy
# After taking all images in ResNet Model and saving the softmax result, ran kmeans on the even image id ones with 
# n clusters = 101 (for each label). However, there were labels being called that didn't make sense. For example, in image ID 0: 
# a person with a face and bookshelf behind it, I would get staple, crocidile, strawberry else. From a color standpoint, I kinda of understand,
# but it would not recognize face as one of the top 5, usually in top 10 with most of the distances being close to 1.
# So, Didn't like some of the results for comparing the image to all cluster's and wanted to try to see if I could do better as I realized
# that n_clusters may not be equilvalent to labels as this k-means is a unsuperviced learning method without labels.
# So, I tried to find the optimal cluster using the silhouette method as I remember using this method in ASU CSE 573 last semester.
# However, talking to Sandipan who did task 2a, I realized that these clusters don't have an automatic map to the label ids. Like the first cluster is not
# guaranteed to be the first label id in the Caltech Database.
# So, I decided to get the cluster of each label id, but instead of finding  one big cluster, I decided to find the optimal number of clusters for each laebl id.
# Why do I approached each label id with multiple centroids - because, every picture will have noise not guranteed to be close. For example an airplane with a lot of stuff in the backgroud
# might be mistaked to a face due to the background of both images, which doesn't make sense. So, with each picture I can pick the closest cluster to that the image we are using for comparision
# to theoretically give the best possible matching labels.

def getEachLabelCentroids(df):
    centroidLists = []
    # Go through each label id
    for i in tqdm(range(0, len(df["LabelID"].unique())), desc="Finding Clusters", ncols=100):
        # Dictionary to keep track of the best one
        maxK = {"K": 0, "SIL": sys.float_info.min, "Centroids": [], "Interia": sys.float_info.min}
        
        # Split the training dataset into even images for training (mentioned in ED Discussion Post)
        # then extract the ones with i's label id and convert to format readable to kmeans package
        traindf = df[df["ImageID"] % 2 == 0]
        traindf = traindf[traindf["LabelID"] == i]
        dataList = [list(row.astype(float)) for row in traindf["ResNet"]]
        
        # Taking too long, so using a counter to stop kinda of like earlystop in Tensorflow
        counter = 0
        
        # Range goes to length of traindf as when doing a fixed like 500, I got error messages saying not enough 
        # samples, so to fix it I use the length of the traindf
        for k in range(2, len(traindf)):
            # Looking at the example graphs, it seems to generally flat line for a long time and 50 makes sure we don't stop too early
            if counter >= 50: 
                break
            else:
                # Using kmeans, get the silhouette score 
                kmeans = sklearn.cluster.KMeans(n_clusters=k, init="k-means++", random_state=0, n_init="auto").fit(dataList)
                # For silhouette score, used euclidean as it was in the articles and it seemed to give good results
                silhouetteScore = sklearn.metrics.silhouette_score(dataList, kmeans.labels_, metric = 'euclidean')
                # Trying to find max, these is how we did this in ASU CSE 110
                if silhouetteScore > maxK["SIL"]:
                    maxK["K"] = k
                    maxK["SIL"] = silhouetteScore
                    maxK["Centroids"] = kmeans.cluster_centers_
                    maxK["Interia"] = kmeans.inertia_
                    # print(k, silScore, maxK["SIL"])
                    # Since there is a change i, counter needs to reset to 0
                    counter = 0
                else:
                    # No update, increase counter so we can stop if there is no changes
                    counter = counter + 1
        # print(maxK)
        # Now finally save the best results to the centroids list for label id i
        centroidLists.append({"LabelID": i, "Centroids": maxK["Centroids"], "Interia": maxK["Interia"], "Silhouette": maxK["SIL"]})
        sleep(1)
        gc.collect()

    # Convert to DataFrame and save results, so it doesn't need to be called all the time
    centerDF = pd.DataFrame(centroidLists)
    centerDF.to_pickle("./database/testFile_2b_centers.pickle")
    # centerDF


def kSimilarImages_2b(image, df, caltechDB, k, resNetWeight, resNetModel, torchDevice):
    # Load the Dataframe
    centerDF = pd.read_pickle("./database/testFile_2b_centers.pickle")

    # Get the ResNet Softmax Result of the image
    result = resNetImageTransformer(image, resNetModel, resNetWeight, torchDevice)
    df_list = []
    # Go through all label ids' centroids
    for index, row in centerDF.iterrows():
        distList = []
        # For each centroid, get the distance for label id i
        for centroid in row["Centroids"]:
            distList.append(scipy.spatial.distance.cosine(result, centroid))
        # Get the min dist as that is the closest for label id i to the image that we are comparing to
        df_list.append({"LabelID": row["LabelID"], "Label": getCalTechCategory(caltechDB, row["LabelID"]), "Distance": min(distList)})

    # Sort and get the top k labels
    result = pd.DataFrame(df_list).sort_values(by="Distance")[0:k].reset_index()[["LabelID", "Label", "Distance"]]

    return result


In [5]:
# This function is the code to generate the resnet
# vectors and exports the database so it can be called in k similar labels
# As a note, task2b and k similar label functions are two differet functions so
# I could test the k similar labels without waiting for the resnet model each time to 
# analyze the images.
def task2b(caltechDB, resNetModel, resNetWeight, torchDevice):
    # Create Pandas Dataframe
    df_list = []

    # For Loop in databases
    for i in tqdm(range(0, len(caltechDB)), desc="Extracting Features", ncols=100):
        # for i in range(1580, 1582, 1):
        # Get Image
        image = caltechDB[i][0]

        # Put image into model and get results
        result = resNetImageTransformer(image, resNetModel, resNetWeight, torchDevice)

        # Attach to List
        row_df = {
            "ImageID": i,
            "Label": getCalTechCategory(caltechDB, caltechDB[i][1]),
            "LabelID": caltechDB[i][1],
            "ResNet": result,
        }
        df_list.append(row_df)

    torch.cuda.empty_cache()

    # Convert List to DataFrame for easy accessability
    df = pd.DataFrame(df_list)
    # print(df.head(5))

    df.to_pickle("./database/testFile_2b.pickle")

In [6]:
# This function gets the reuqired user input for the functions
def userInput():
    IMAGE_INPUT = input(
    """
    Provide one of the following:
    1. An Image ID in the Caltech101 dataset in range [0, 8676].
    2. The name of an image file in /Code/input/ directory (eg: image.jpg).
    """
    )

    kLabels = int(input("Enter K, the number of similar labels to find for ResNet model."))

    image = get_query_image(IMAGE_INPUT)
    print("User inputed image ID / image: ", IMAGE_INPUT)
    display(image)
    return image, kLabels



In [7]:
# Next 3 cells are the main code that includes the user input, then calls task2b, then the k similar labels function

# Declare the Global Variables

# Load Database
caltechDB = torchvision.datasets.Caltech101("./database/CaltechDB/", download=True, target_type="category")

# Get Device
torchDevice = getDevice()

# Get ResNet Model
resNetWeight, resNetModel = getResNetModel(torchDevice)

# Get User Inputs
# image, kLabels = get_query_image("0"),10 #For Testing
image, kLabels = userInput()

Files already downloaded and verified
Using Device:  cuda
Files already downloaded and verified


In [8]:
# Get the vectors for the images
# task2b(caltechDB, resNetModel, resNetWeight, torchDevice)

In [24]:
# Get K Similar Images based on an image
df = pd.read_pickle("./database/testFile_2b.pickle")
kSimilarImages_2b(image, df, caltechDB, kLabels, resNetWeight, resNetModel, torchDevice)

,LabelID,Label,Distance
0,0,Faces_2,0.087120
1,1,Faces_3,0.251100
2,43,garfield,0.809787
3,83,snoopy,0.815207
4,13,brain,0.824121
5,42,flamingo_head,0.830275
6,32,dollar_bill,0.837174
7,9,bass,0.840410
8,73,platypus,0.841067
9,67,octopus,0.841398
